In [1]:
import sys, importlib, json
from pathlib import Path
import pandas as pd

# Locate local module: model/standardize_makes.py
candidates = [Path.cwd() / "model", Path.cwd().parent / "model"]
for p in candidates:
    if (p / "standardize_makes.py").exists():
        sys.path.insert(0, str(p))
        break
else:
    raise FileNotFoundError("standardize_makes.py not found in ./model or ../model")

# Reload cleanly and import
sys.modules.pop("standardize_makes", None)
sm = importlib.import_module("standardize_makes")
canonicalize_make = sm.canonicalize_make

# Find the data file whether you launched from repo root or notebooks/
DATA_IN = "data/clean_used_cars.csv" if (Path.cwd()/"data/clean_used_cars.csv").exists() else str(Path.cwd().parent/"data/clean_used_cars.csv")
print("Using data:", DATA_IN)

df = pd.read_csv(DATA_IN, low_memory=False)
assert "make" in df.columns, "Column 'make' not found."
print(f"Rows: {len(df):,} | Unique makes BEFORE: {df['make'].nunique():,}")

Using data: /workspaces/used-car-price-app/data/clean_used_cars.csv
Rows: 116,624 | Unique makes BEFORE: 367


In [2]:
import re

EXTRA_SYNONYMS = {
    # direct fixes from your list
    "mercedes-b": "Mercedes-Benz",
    "mercedes benz": "Mercedes-Benz",
    "mercedes": "Mercedes-Benz",
    "volkswagon": "Volkswagen",
    "vw": "Volkswagen",
    "pontlac": "Pontiac",
    "piymouth": "Plymouth",
    "chrylser": "Chrysler",
    "crhysler": "Chrysler",
    "infinity": "INFINITI",
    "landrover": "Land Rover",
    "rolls royce": "Rolls-Royce",
    "citroen": "Citroën",
    "citroën": "Citroën",
    "studabaker": "Studebaker",
    "willy's jeep": "Willys",
    "american motors": "AMC",
    "amc / rambler": "AMC",
    "nissan datsun": "Nissan",
    "gmc aka chevy express": "GMC",
    "freightlnr": "Freightliner",
    "international": "International Harvester",
    "international harvester": "International Harvester",
    # catch lines that are obviously titles rather than makes -> pick a brand token
    # (handled below in fallback matcher)
}

# brand tokens to search anywhere in the string as a fallback
BRAND_TOKENS = {
    # common car brands; add more if needed
    r"\bbmw\b": "BMW",
    r"\bmercedes\b|\bbenz\b": "Mercedes-Benz",
    r"\bvolkswagen\b|\bvw\b": "Volkswagen",
    r"\bchevy\b|\bchevrolet\b": "Chevrolet",
    r"\bford\b": "Ford",
    r"\btoyota\b": "Toyota",
    r"\bdodge\b": "Dodge",
    r"\bjeep\b": "Jeep",
    r"\bcadillac\b": "Cadillac",
    r"\bgmc\b": "GMC",
    r"\bhonda\b": "Honda",
    r"\bnissan\b": "Nissan",
    r"\baudi\b": "Audi",
    r"\bchrysler\b": "Chrysler",
    r"\bjaguar\b": "Jaguar",
    r"\bsubaru\b": "Subaru",
    r"\bvolvo\b": "Volvo",
    r"\bhyundai\b": "Hyundai",
    r"\blexus\b": "Lexus",
    r"\bmazda\b": "Mazda",
    r"\bbuick\b": "Buick",
    r"\bporsche\b": "Porsche",
    r"\bsaab\b": "Saab",
    r"\bscion\b": "Scion",
    r"\bsmart\b": "smart",
    r"\bmini\b": "MINI",
    r"\btesla\b": "Tesla",
    r"\bram\b": "RAM",
    r"\bfiat\b": "FIAT",
    r"\bgenesis\b": "Genesis",
    r"\bkia\b": "Kia",
    r"\bmitsubishi\b": "Mitsubishi",
    r"\binfiniti\b|\binfinity\b": "INFINITI",
    r"\bland\s*rover\b|rangerover|range rover": "Land Rover",
    r"\bro lls[\s-]?royce\b|rolls royce|rolls-royce": "Rolls-Royce",
    r"\bamerican\s+motors\b|\bamc\b": "AMC",
    r"\bwillys\b": "Willys",
    r"\bstudebaker\b|studabaker": "Studebaker",
    r"\binternational\b|international harvester": "International Harvester",
    r"\bcitroen\b|citro[eë]n": "Citroën",
}

def clean_make_value(x: str) -> str:
    """Use overrides → your canonicalizer → token fallback."""
    if pd.isna(x) or str(x).strip() == "":
        return pd.NA
    s = str(x).strip().lower()
    # Exact override first (matches simple/raw strings in your list)
    if s in EXTRA_SYNONYMS:
        return EXTRA_SYNONYMS[s]
    # Try your robust function
    can = canonicalize_make(x)
    if isinstance(can, str) and can:
        return can
    # Fallback: look for brand tokens inside noisy strings (titles)
    for rx, brand in BRAND_TOKENS.items():
        if re.search(rx, s):
            return brand
    # Otherwise: title-case fallback
    return str(x).strip().title()

In [3]:
df_tmp = df.copy()
before_unique = df_tmp["make"].nunique(dropna=False)
df_tmp["make_fixed"] = df_tmp["make"].apply(clean_make_value)
after_unique = df_tmp["make_fixed"].nunique(dropna=False)

print(f"Unique makes: {before_unique} → {after_unique}")

# What changed?
chg = (
    df_tmp.assign(make_raw=df_tmp["make"])
          .loc[lambda d: d["make_raw"].astype(str).str.lower() != d["make_fixed"].astype(str).str.lower(),
               ["make_raw","make_fixed"]]
          .value_counts()
          .reset_index(name="count")
          .sort_values("count", ascending=False)
)
chg.head(30)

Unique makes: 367 → 248


,make_raw,make_fixed,count
0,replica/kit makes,Replica Kit Makes,187
1,international,International Harvester,38
2,american motors,AMC,11
3,mercedes benz,Mercedes-Benz,11
4,austin-healey,Austin Healey,6
5,chevy,Chevrolet,5
6,vw,Volkswagen,5
7,citroen,Citroën,4
8,toyota one owner,Toyota,4
9,mercedes,Mercedes-Benz,4


In [4]:
summary = (
    df_tmp.assign(make_raw=df_tmp["make"].astype(str))
          .groupby("make_fixed")
          .agg(total=("make_fixed","size"),
               sample_variants=("make_raw", lambda s: ", ".join(sorted(set(v for v in s.str.split().str[0].str.title()) - {s.name})[:8])))
          .sort_values("total", ascending=False)
          .reset_index()
          .rename(columns={"make_fixed":"make"})
)

summary.head(20)

,make,total,sample_variants
0,Ford,21110,Ford
1,Chevrolet,20254,"Cheverolet, Chevrolet, Chevy"
2,Toyota,6449,Toyota
3,Mercedes-Benz,6104,"Mercedes, Mercedes-B, Mercedes-Benz"
4,Dodge,5603,"Dodge, Immaculate"
5,BMW,5033,Bmw
6,Jeep,4437,Jeep
7,Cadillac,3576,Cadillac
8,Volkswagen,3415,"Volkswagen, Volkswagon, Vw"
9,Honda,3234,Honda


In [5]:
# Save organized summary for your review (view in VS Code explorer)
OUT_DIR = Path(DATA_IN).parent
summary.to_csv(OUT_DIR / "makes_canonical_summary.csv", index=False)
print("Wrote:", OUT_DIR / "makes_canonical_summary.csv")

# If the audit looks good, persist to your dataset:
df["make"] = df_tmp["make_fixed"]
df.to_csv(DATA_IN, index=False)
print("✅ Saved cleaned makes back to", DATA_IN)

Wrote: /workspaces/used-car-price-app/data/makes_canonical_summary.csv
✅ Saved cleaned makes back to /workspaces/used-car-price-app/data/clean_used_cars.csv


In [6]:
# BMW/Mercedes/VW should be clean
print("BMW values:", df.loc[df["make"].str.contains("bmw", case=False, na=False), "make"].unique())
print("Mercedes values:", df.loc[df["make"].str.contains("mercedes|benz", case=False, na=False), "make"].unique())
print("VW values:", df.loc[df["make"].str.contains("volk|\\bvw\\b", case=False, na=False), "make"].unique())

# Top-15 brands
df["make"].value_counts().head(15)

BMW values: ['BMW']
Mercedes values: ['Mercedes-Benz']
VW values: ['Volkswagen']


make
Ford             21110
Chevrolet        20254
Toyota            6449
Mercedes-Benz     6104
Dodge             5603
BMW               5033
Jeep              4437
Cadillac          3576
Volkswagen        3415
Honda             3234
Pontiac           2569
GMC               2460
Nissan            2395
Porsche           2200
Lincoln           1817
Name: count, dtype: int64